# QVAR Identification: Restriction Convention Guide

This notebook explains **exactly** how to code zero and sign restrictions
for the Bayesian QVAR framework.

---

## The convention

Both zero and sign restrictions use the **same key format**:

```python
(shock_idx, response_var_idx, horizon): value
```

| Position | Meaning | Example |
|----------|---------|----------|
| `shock_idx` | Which structural shock (0-indexed) | `0` = first shock |
| `response_var_idx` | Which variable responds (0-indexed) | `1` = second variable |
| `horizon` | At which IRF horizon | `0` = impact, `4` = 1 year |
| **value** | The restriction type | `0` = zero, `+1` = positive, `-1` = negative |

**Read it as:** *"The response of variable `response_var_idx` to shock `shock_idx` at horizon `horizon` is [zero / positive / negative]."*

---

## Setup: A 4-variable system

Let's work with a standard macro-finance VAR:

| Index | Variable | Mnemonic |
|-------|----------|----------|
| 0 | Excess Bond Premium | EBP |
| 1 | GDP growth | GDP |
| 2 | CPI inflation | CPI |
| 3 | Policy rate | Rate |

And we want to identify **4 structural shocks**:

| Shock index | Interpretation |
|-------------|----------------|
| 0 | Financial shock |
| 1 | Demand shock |
| 2 | Supply shock |
| 3 | Monetary policy shock |

In [ ]:
# Variable indices — just for readability
EBP  = 0
GDP  = 1
CPI  = 2
RATE = 3

# Shock indices
FINANCIAL = 0
DEMAND    = 1
SUPPLY    = 2
MP        = 3  # monetary policy

---

## Example 1: Pure sign restrictions

**Story:** A financial shock tightens credit (EBP up) and lowers GDP.
A demand shock raises both GDP and CPI. A supply shock raises GDP
but lowers CPI. A monetary policy tightening raises the rate and
lowers GDP.

All restrictions are on **impact** (horizon = 0).

| | Financial | Demand | Supply | MP |
|------|-----------|--------|--------|-----|
| EBP | **+** | | | |
| GDP | **−** | **+** | **+** | **−** |
| CPI | | **+** | **−** | |
| Rate | | | | **+** |

In [ ]:
sign_restrictions_ex1 = {
    # Financial shock (shock 0)
    (FINANCIAL, EBP, 0): +1,   # EBP rises on impact
    (FINANCIAL, GDP, 0): -1,   # GDP falls on impact

    # Demand shock (shock 1)
    (DEMAND, GDP, 0): +1,      # GDP rises
    (DEMAND, CPI, 0): +1,      # CPI rises

    # Supply shock (shock 2)
    (SUPPLY, GDP, 0): +1,      # GDP rises
    (SUPPLY, CPI, 0): -1,      # CPI falls (supply expansion)

    # Monetary policy shock (shock 3)
    (MP, RATE, 0): +1,         # Rate rises (tightening)
    (MP, GDP,  0): -1,         # GDP falls
}

print("Example 1: Pure sign restrictions")
print(f"  {len(sign_restrictions_ex1)} sign constraints")
for (s, v, h), val in sorted(sign_restrictions_ex1.items()):
    names = ['EBP', 'GDP', 'CPI', 'Rate']
    shocks = ['Financial', 'Demand', 'Supply', 'MP']
    sign_str = '+' if val > 0 else '−'
    print(f"  {shocks[s]:>10} shock → {names[v]:<6} at h={h}: {sign_str}")

To use this:
```python
model.configure(
    lags=2,
    identification="sign_restrictions",
    sign_restrictions=sign_restrictions_ex1,
)
```

---

## Example 2: Zero + sign restrictions

**Story:** Same as above, but now we also impose:
- Financial shocks do NOT affect the policy rate contemporaneously
  (the central bank reacts with a lag)
- Monetary policy shocks do NOT affect CPI contemporaneously
  (prices are sticky at impact)

| | Financial | Demand | Supply | MP |
|------|-----------|--------|--------|-----|
| EBP | **+** | | | |
| GDP | **−** | **+** | **+** | **−** |
| CPI | | **+** | **−** | **0** |
| Rate | **0** | | | **+** |

(**0** = zero restriction, **+/−** = sign restriction, blank = unrestricted)

In [ ]:
# ZERO restrictions (value = 0)
zero_restrictions_ex2 = {
    (FINANCIAL, RATE, 0): 0,  # Financial shock has NO impact on Rate at h=0
    (MP,        CPI,  0): 0,  # MP shock has NO impact on CPI at h=0
}

# SIGN restrictions (value = +1 or -1)
sign_restrictions_ex2 = {
    (FINANCIAL, EBP, 0): +1,
    (FINANCIAL, GDP, 0): -1,

    (DEMAND, GDP, 0): +1,
    (DEMAND, CPI, 0): +1,

    (SUPPLY, GDP, 0): +1,
    (SUPPLY, CPI, 0): -1,

    (MP, RATE, 0): +1,
    (MP, GDP,  0): -1,
}

print("Example 2: Zero + sign restrictions")
print(f"  {len(zero_restrictions_ex2)} zero constraints")
print(f"  {len(sign_restrictions_ex2)} sign constraints")

To use this:
```python
model.configure(
    lags=2,
    identification="zero_sign_restrictions",
    zero_restrictions=zero_restrictions_ex2,
    sign_restrictions=sign_restrictions_ex2,
)
```

---

## Example 3: Zeros at different horizons

**Story:** 3-variable system `[Output, Prices, Oil]`.
- **Oil supply shock**: raises oil prices, lowers output
- **Demand shock**: raises both output and prices, but has NO effect
  on oil at horizon 0 (oil supply is inelastic in the short run)
- **Technology shock**: has zero long-run effect on prices (monetary
  neutrality), specifically zero effect on prices at h=20

| | Oil supply (0) | Demand (1) | Technology (2) |
|--------|----------------|------------|----------------|
| Output | **−** | **+** | **+** |
| Prices | | **+** | **0 at h=20** |
| Oil | **+** | **0 at h=0** | |

In [ ]:
OUTPUT = 0
PRICES = 1
OIL    = 2

OIL_SUPPLY = 0
DEMAND_3   = 1
TECH       = 2

zero_restrictions_ex3 = {
    (DEMAND_3, OIL,    0):  0,  # Demand shock → zero impact on Oil at h=0
    (TECH,     PRICES, 20): 0,  # Tech shock → zero effect on Prices at h=20
}

sign_restrictions_ex3 = {
    (OIL_SUPPLY, OUTPUT, 0): -1,  # Oil shock lowers output
    (OIL_SUPPLY, OIL,    0): +1,  # Oil shock raises oil price

    (DEMAND_3, OUTPUT, 0): +1,    # Demand raises output
    (DEMAND_3, PRICES, 0): +1,    # Demand raises prices

    (TECH, OUTPUT, 0): +1,        # Tech raises output
}

print("Example 3: Zeros at different horizons")
print(f"  Zero: Demand→Oil at h=0, Tech→Prices at h=20")
print(f"  {len(sign_restrictions_ex3)} sign constraints")

Note the **long-run zero restriction** `(TECH, PRICES, 20): 0`. This
constrains the cumulative IRF at horizon 20, which approximates a
long-run restriction à la Blanchard & Quah (1989).

```python
model.configure(
    lags=4,
    identification="zero_sign_restrictions",
    zero_restrictions=zero_restrictions_ex3,
    sign_restrictions=sign_restrictions_ex3,
)
```

---

## Example 4: Block-recursive structure via zeros only

**Story:** 3-variable system `[EBP, GDP, Rate]`.
EBP is ordered first and treated as exogenous — the other
two shocks cannot affect it on impact. This is like a partial
Cholesky but without imposing a full recursive ordering between
GDP and Rate.

| | EBP shock (0) | GDP shock (1) | Rate shock (2) |
|------|---------------|---------------|----------------|
| EBP | free | **0** | **0** |
| GDP | free | free | free |
| Rate | free | free | free |

In [ ]:
EBP_  = 0
GDP_  = 1
RATE_ = 2

# Only zero restrictions — no sign restrictions
zero_restrictions_ex4 = {
    (1, EBP_, 0): 0,   # GDP shock has no contemporaneous effect on EBP
    (2, EBP_, 0): 0,   # Rate shock has no contemporaneous effect on EBP
}

sign_restrictions_ex4 = {}  # No signs needed

print("Example 4: Block-recursive via zeros only")
print("  EBP is block-exogenous: shocks 1 and 2 cannot affect it at h=0")
print("  GDP and Rate are free to affect each other contemporaneously")

```python
model.configure(
    lags=2,
    identification="zero_sign_restrictions",
    zero_restrictions=zero_restrictions_ex4,
    sign_restrictions=sign_restrictions_ex4,
)
```

---

## Example 5: Sign restrictions across multiple horizons

**Story:** 3-variable system `[EBP, GDP, FCI]`.
We want the financial shock to have a **persistent** negative
effect on GDP — not just at impact, but also at h=1, 2, 3, 4.

This makes identification much sharper (more restrictions = smaller
identified set), but also harder to satisfy (lower acceptance rate).

In [ ]:
sign_restrictions_ex5 = {
    # Financial shock (0): EBP up, GDP down persistently
    (0, 0, 0): +1,    # EBP rises at h=0
    (0, 1, 0): -1,    # GDP falls at h=0
    (0, 1, 1): -1,    # GDP still negative at h=1
    (0, 1, 2): -1,    # GDP still negative at h=2
    (0, 1, 3): -1,    # GDP still negative at h=3
    (0, 1, 4): -1,    # GDP still negative at h=4 (one year)
    (0, 2, 0): +1,    # FCI tightens at h=0
}

print("Example 5: Multi-horizon sign restrictions")
print("  Financial shock must lower GDP for 5 consecutive quarters")
print(f"  {len(sign_restrictions_ex5)} total constraints")
print("  Note: set max_horizon_check=4 in estimate_and_analyze()")

When using multi-horizon restrictions, set `max_horizon_check`
to the maximum horizon you restrict:

```python
model.estimate_and_analyze(
    ...
    max_horizon_check=4,  # Must check up to h=4
)
```

---

## Example 6: Full JIMF paper setup

The Beutel et al. (2025) paper uses a **Cholesky** identification
with the EBP ordered first. This is equivalent to:

| | EBP shock (0) | FCI shock (1) | GDP shock (2) |
|-----|---------------|---------------|---------------|
| EBP | free | **0** | **0** |
| FCI | free | free | **0** |
| GDP | free | free | free |

A full Cholesky = zeros everywhere above the diagonal.

In [ ]:
# Cholesky via zero restrictions (equivalent to identification="cholesky")
zero_restrictions_cholesky = {
    (1, 0, 0): 0,   # Shock 1 → no impact on var 0
    (2, 0, 0): 0,   # Shock 2 → no impact on var 0
    (2, 1, 0): 0,   # Shock 2 → no impact on var 1
}

print("Example 6: Cholesky = full set of zero restrictions")
print("  This gives the same result as identification='cholesky'")
print("  but you could relax one zero to get a partial ordering")

---

## Quick reference

```
restriction_key = (shock_idx, response_var_idx, horizon)
                     ↓              ↓               ↓
               Which shock    Which variable    At what lag
               causes it?     is affected?      (0=impact)
```

| Dict | Key format | Value | Meaning |
|------|-----------|-------|----------|
| `zero_restrictions` | `(shock, var, h): 0` | `0` | IRF is exactly zero |
| `sign_restrictions` | `(shock, var, h): +1` | `+1` | IRF must be positive |
| `sign_restrictions` | `(shock, var, h): -1` | `-1` | IRF must be negative |

**Reading example:**
```python
(0, 1, 0): -1
```
*"Shock 0 decreases variable 1 on impact."*

```python
(2, 0, 4): 0
```
*"Shock 2 has zero effect on variable 0 at horizon 4."*

---

## Running the examples

Here is a complete working example using Example 2 (zero + sign):

In [ ]:
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, '..')

# --- Generate synthetic 4-variable data ---
np.random.seed(42)
T = 200
k = 4
data = np.random.randn(T, k) * 0.3
B = np.array([
    [0.5, 0.1, 0.0, 0.0],
    [-0.2, 0.4, 0.1, -0.1],
    [0.0, 0.1, 0.3, 0.1],
    [0.1, -0.1, 0.0, 0.4],
])
for t in range(1, T):
    data[t] += B @ data[t-1]

dates = pd.date_range('1980Q1', periods=T, freq='QE')
df = pd.DataFrame(data, columns=['EBP', 'GDP', 'CPI', 'Rate'], index=dates)

# --- Configure and run ---
from src.econometrics.qvar_wrapper import QVARModel

model = QVARModel()
model.set_data(df)
model.select_variables(['EBP', 'GDP', 'CPI', 'Rate'])

model.configure(
    lags=2,
    identification="zero_sign_restrictions",
    zero_restrictions={
        (0, 3, 0): 0,  # Financial shock → no impact on Rate
        (3, 2, 0): 0,  # MP shock → no impact on CPI
    },
    sign_restrictions={
        (0, 0, 0): +1,  # Financial shock raises EBP
        (0, 1, 0): -1,  # Financial shock lowers GDP
        (1, 1, 0): +1,  # Demand shock raises GDP
        (1, 2, 0): +1,  # Demand shock raises CPI
        (2, 1, 0): +1,  # Supply shock raises GDP
        (2, 2, 0): -1,  # Supply shock lowers CPI
        (3, 3, 0): +1,  # MP shock raises Rate
        (3, 1, 0): -1,  # MP shock lowers GDP
    },
)

model.estimate_and_analyze(
    quantiles=[0.1, 0.5, 0.9],
    shock_idx=0,
    horizon=12,
    n_draws=500,
    n_burnin=200,
    n_rotations=200,
    verbose=True,
)

print(model.summary())